In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_ollama.llms import OllamaLLM
import os

In [2]:

# Function to read the markdown table from "table.md"
def read_markdown_table(filepath):
    # print("Checking if the file exists...")  # Debug print
    try:
        # Check if file exists
        if not os.path.exists(filepath):
            return f"File not found: {filepath}"
        
        # print("File exists, checking readability...")  # Debug print
        # Check if file is readable
        if not os.access(filepath, os.R_OK):
            return f"File is not readable: {filepath}"
        
        # print("Reading the content of the file...")  # Debug print
        # Read the content of the file
        with open(filepath, 'r') as file:
            markdown_content = file.read()
        
        return markdown_content

    except Exception as e:
        # Return a more detailed error message
        return f"Error reading file {filepath}: {str(e)}"

# Example usage
markdown_filepath = "/home/aiml/CredirScoring/table.md"
markdown_table = read_markdown_table(markdown_filepath)

# Print the table content or the error message
# print("Markdown Table Content:")  # Debug print
# print(markdown_table)


In [3]:
class AgentCall:
    def __init__(self, markdown_output):
        self.markdown_output = markdown_output

        self.template = """
            Given this markdown table of financial particulars:

            {markdown_output}, 

            1. Match the following items with the terms of the table in every new line. For each item, if found or exactly synonymous, 
            return the result in this JSON format:

            [
            {{
                "item": "{item_list}",
                "value": "<particulars>"
            }},
            ...
            ]

            2. If the item could not match the criteria of Condition 1, 
            check if an item can be derived by summing up parts, return it like this:

            [
            {{
                "item": "{item_list}",
                "value": ["<particulars>", "<particulars>"]
            }},
            ...
            ]

            3. If an item is neither found nor derivable, return it as:

            [
            {{
                "item": "{item_list}",
                "value": None
            }},
            ...

            NOTE: DO this for each of the item in {item_list}

            
            """

        # Create the prompt template with proper input variables
        self.prompt = PromptTemplate(
            input_variables=["markdown_output", "item_list"],
            template=self.template)

        # Model initialization (ensure this model class is correct and imported)
        model = OllamaLLM(model="gpt-4")

        # Define the chain that will process the prompt and invoke the model
        self.chain = self.prompt | model

    def generate_response(self, items):
        # Prepare the input mapping (dictionary) for the prompt
        formatted_prompt = {
            "markdown_output": self.markdown_output, 
            "item_list": ", ".join(items)
        }

        try:
            # Send the formatted prompt to the LLM for response
            response = self.chain.invoke(formatted_prompt)
            return response
        except Exception as e:
            return f"Error generating response: {e}"


In [4]:
class InputList:
    def __init__(self):
        self.items = ['total assets', 'total liabilities', 'current assets', 'current liabilities', 'intangible assets',
                   'long term loan-current portion', 'inventories', 'accounts receivables', 'capital work in progress', 
                   'non-current assets','revenue', 'net profit after tax', 'operating profit', 'earnings before tax',
                   'finance cost','cost of sales', 'cash and cash equivalents at end of year', 
                   'cash flows from operating activities', 'cash flows from financing activities']

In [5]:

# Initialize the AgentCall class with the markdown table content
agent = AgentCall(markdown_table)

# Initialize the InputList and send all items at once to the agent
input_list = InputList()
response = agent.generate_response(input_list.items)

# Print the response
print(response)

Error generating response: model "gpt-4" not found, try pulling it first


In [6]:
# chain.invoke({"question": "what is langchain?"})